# 03 — Pitch Charts
**AirSense Cameroon · IndabaX 2026**

**Input:**  `../data/cameroon_official_dataset.csv` + trained models
**Output:** PNG charts ready for the pitch deck (saved in `notebooks/`)

Run after 02_Model_Training.ipynb.


In [ ]:
import subprocess, sys
pkgs = ["pandas","numpy","matplotlib","seaborn","plotly","openpyxl",
        "scikit-learn","xgboost","shap","joblib"]
for p in pkgs:
    try: __import__(p.replace("-","_").split("==")[0])
    except ImportError:
        subprocess.check_call([sys.executable,"-m","pip","install",p,"-q"])
print("All packages ready")


In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import json, os
import warnings; warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi":150,"axes.spines.top":False,"axes.spines.right":False})

for path in ["../data/cameroon_official_dataset.csv","data/cameroon_official_dataset.csv"]:
    if os.path.exists(path): DATA_PATH=path; break

df = pd.read_csv(DATA_PATH, parse_dates=["date"])
print(f"Loaded: {df.shape[0]:,} rows · {df['city'].nunique()} cities")

for path in ["../models/rl_thresholds.json","models/rl_thresholds.json"]:
    if os.path.exists(path):
        with open(path) as f: rl = json.load(f)
        print(f"RL thresholds: {len(rl)} cities"); break

for path in ["../models/model_meta.json","models/model_meta.json"]:
    if os.path.exists(path):
        with open(path) as f: meta = json.load(f)
        print(f"Model R2={meta['metrics']['r2']}"); break

RCOLS={"Far North":"#c084fc","North":"#f87171","Adamawa":"#f97316","Littoral":"#fb923c",
       "Centre":"#fbbf24","West":"#a3e635","North West":"#4ade80",
       "South West":"#34d399","East":"#2dd4bf","South":"#1abc9c"}
OUT = os.path.dirname(DATA_PATH).replace("data","notebooks")
os.makedirs(OUT, exist_ok=True)
print(f"Charts will be saved to: {OUT}")


## Slide 1 — The problem stat

In [ ]:
above_who = (df["pm25_proxy"] > 35).mean() * 100
fig, ax = plt.subplots(figsize=(10,3))
ax.text(0.5,0.6,f"{above_who:.0f}%",transform=ax.transAxes,
        fontsize=72,fontweight="bold",ha="center",va="center",color="#e05c2a")
ax.text(0.5,0.18,"of Cameroonian city-days exceed WHO PM2.5 safe limits",
        transform=ax.transAxes,fontsize=16,ha="center",color="#1a3c5e")
ax.axis("off")
plt.tight_layout()
plt.savefig(f"{OUT}/slide_01_problem_stat.png",bbox_inches="tight",facecolor="white")
plt.show()
print(f"Stat: {above_who:.0f}% | Far North avg: {df[df['region']=='Far North']['pm25_proxy'].mean():.0f} ug/m3")


## Slide 2 — North-South divide

In [ ]:
region_avg=df.groupby("region")["pm25_proxy"].mean().sort_values(ascending=False)
colors=[RCOLS.get(r,"#888") for r in region_avg.index]
fig,ax=plt.subplots(figsize=(12,4))
bars=ax.bar(region_avg.index,region_avg.values,color=colors,alpha=0.88,edgecolor="white")
ax.axhline(35,color="#ff7e00",linestyle="--",linewidth=2,label="WHO threshold 35 ug/m3")
ax.set_title("PM2.5 by Region — Region-specific thresholds are essential",fontweight="bold",pad=10)
ax.set_ylabel("Mean PM2.5 (ug/m3)")
for bar,val in zip(bars,region_avg.values):
    ax.text(bar.get_x()+bar.get_width()/2,val+0.3,f"{val:.0f}",ha="center",fontsize=9)
ax.legend(); plt.xticks(rotation=15,ha="right"); plt.tight_layout()
plt.savefig(f"{OUT}/slide_02_north_south.png",bbox_inches="tight",facecolor="white")
plt.show()


## Slide 3 — System architecture

In [ ]:
fig,ax=plt.subplots(figsize=(13,3.5))
ax.axis("off")
boxes=[
    (0.03,"Open-Meteo
Weather API
(42 cities)","#1a3c5e","white"),
    (0.21,"Feature
Engineering
(27 features)","#4a6fa5","white"),
    (0.40,"XGBoost +
REINFORCE RL
Thresholds","#e05c2a","white"),
    (0.59,"FastAPI
10 Endpoints
(EN/FR)","#2ecc71","white"),
    (0.78,"Streamlit
Dashboard
+ PWA","#9b59b6","white"),
]
for x,label,bg,fg in boxes:
    rect=mpatches.FancyBboxPatch((x,0.15),0.17,0.70,
                                  boxstyle="round,pad=0.02",
                                  facecolor=bg,edgecolor="white",linewidth=2,
                                  transform=ax.transAxes)
    ax.add_patch(rect)
    ax.text(x+0.085,0.5,label,transform=ax.transAxes,
            ha="center",va="center",fontsize=9,color=fg,fontweight="bold")
    if x<0.78:
        ax.annotate("",xy=(x+0.185,0.5),xytext=(x+0.17,0.5),xycoords="axes fraction",
                    arrowprops=dict(arrowstyle="->",color="#888",lw=2))
ax.set_title("AirSense Cameroon — System Architecture",fontsize=14,color="#1a3c5e",pad=12)
r2=meta["metrics"]["r2"]; mae=meta["metrics"]["mae"]
ax.text(0.5,0.05,f"R2={r2}  MAE={mae} ug/m3  |  42 cities · 10 regions · Bilingual EN/FR · ESP32 IoT",
        transform=ax.transAxes,ha="center",fontsize=9,color="#555")
plt.tight_layout()
plt.savefig(f"{OUT}/slide_03_architecture.png",bbox_inches="tight",facecolor="white")
plt.show()


## Slide 4 — Model performance

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(12,4))
models=["Random Forest","XGBoost"]
r2s=[meta["baseline"]["r2"],meta["metrics"]["r2"]]
maes=[meta["baseline"]["mae"],meta["metrics"]["mae"]]
colors=["#4a6fa5","#e05c2a"]
axes[0].bar(models,r2s,color=colors,alpha=0.85,edgecolor="white",width=0.5)
axes[0].set_ylim(0.9,1.0); axes[0].set_ylabel("R2 Score")
axes[0].set_title("R2 Score — Higher is better",fontweight="bold")
for i,(m,v) in enumerate(zip(models,r2s)):
    axes[0].text(i,v+0.001,f"{v:.4f}",ha="center",fontsize=11,fontweight="bold")
axes[1].bar(models,maes,color=colors,alpha=0.85,edgecolor="white",width=0.5)
axes[1].set_ylabel("MAE (ug/m3)"); axes[1].set_title("MAE — Lower is better",fontweight="bold")
for i,(m,v) in enumerate(zip(models,maes)):
    axes[1].text(i,v+0.01,f"{v:.4f}",ha="center",fontsize=11,fontweight="bold")
plt.suptitle("Baseline vs Advanced Model",fontweight="bold",y=1.02)
plt.tight_layout()
plt.savefig(f"{OUT}/slide_04_model_performance.png",bbox_inches="tight",facecolor="white")
plt.show()


## Slide 5 — RL thresholds vs WHO

In [ ]:
rl_df=pd.DataFrame(rl).T.reset_index()
rl_df.columns=["city","region","threshold","who","multiplier","baseline"]
rl_df["threshold"]=pd.to_numeric(rl_df["threshold"])
rl_df=rl_df.sort_values("threshold",ascending=False)
colors=[RCOLS.get(r,"#888") for r in rl_df["region"]]
fig,ax=plt.subplots(figsize=(14,5))
ax.bar(range(len(rl_df)),rl_df["threshold"],color=colors,alpha=0.85,width=0.7)
ax.axhline(35,color="#ff7e00",linestyle="--",linewidth=2,label="WHO standard 35 ug/m3")
ax.set_xticks(range(len(rl_df)))
ax.set_xticklabels(rl_df["city"],rotation=75,ha="right",fontsize=8)
ax.set_ylabel("Alert Threshold (ug/m3)")
ax.set_title("REINFORCE RL: City-Specific Thresholds vs WHO Standard
WHO of 35 would fire daily in Far North — RL learns the real anomaly level",fontweight="bold")
patches=[mpatches.Patch(color=c,label=r) for r,c in RCOLS.items() if r in rl_df["region"].values]
ax.legend(handles=patches+[plt.Line2D([0],[0],color="#ff7e00",linestyle="--",label="WHO 35")],
          loc="upper right",fontsize=7,ncol=2)
plt.tight_layout()
plt.savefig(f"{OUT}/slide_05_rl_thresholds.png",bbox_inches="tight",facecolor="white")
plt.show()


In [ ]:
import glob
charts = sorted(glob.glob(f"{OUT}/slide_*.png"))
print(f"Generated {len(charts)} pitch charts:")
for c in charts:
    print(f"  {os.path.basename(c)}")
print("\nPitch deck charts ready.")
